In [ ]:
# Cell 1: install (uncomment if needed) and load PaliGemma
# Run this cell once at the top of the notebook.

# (Uncomment the install lines when running in a fresh environment)
# !pip install -q "transformers>=4.35.0" accelerate torch pillow

import torch
from PIL import Image
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

MODEL = "google/paligemma2-3b-mix-224"   # change if you prefer a different checkpoint

print("Loading model:", MODEL)
try:
    # Try to load with auto device placement and bfloat16 if supported
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        MODEL,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    processor = AutoProcessor.from_pretrained(MODEL)
    print("Loaded on device(s):", model.device_map if hasattr(model, "device_map") else model.device)
except Exception as e:
    print("Auto device_map / bfloat16 load failed — falling back to CPU:", str(e))
    model = PaliGemmaForConditionalGeneration.from_pretrained(MODEL, device_map={"": "cpu"})
    processor = AutoProcessor.from_pretrained(MODEL)

# small helper
def generate_from_image(image, prompt, max_new_tokens=60):
    """
    Utility to run inference: returns decoded generated string.
    """
    # processor accepts image + text
    inputs = processor(image, prompt, return_tensors="pt")
    # move tensors to model device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, cache_implementation="static")
    return processor.decode(out[0], skip_special_tokens=True)

print("Model and processor ready.")


In [ ]:
# Cell 2: test inference
# Replace IMAGE_PATH with your local file or a URL (PIL can open a URL if you fetch it).
IMAGE_PATH = "./data/LLaVA-Med/images/test/1.jpeg"   # <-- change to your image file path

prompt = """You are a clinical diagnostic assistant. Analyze the provided image for signs of an
            Adverse Drug Effect (ADE). Identify the most likely body location. The image is: <image>"""

# Load the image (PIL)
try:
    image = Image.open(IMAGE_PATH).convert("RGB")
except FileNotFoundError:
    raise FileNotFoundError(f"Image not found at {IMAGE_PATH} — update IMAGE_PATH to a valid file")

print("Running model... (this can take a few seconds on GPU, much longer on CPU)")
result = generate_from_image(image, prompt, max_new_tokens=80)
print("=== MODEL OUTPUT ===")
print(result)
